# Skip-gram Softmax 概率计算 P(o|c)> CS224N 2025 L01 Word Vectors — Code Capsule: skipgram-softmax>> **这段代码在看什么**：用一个小词汇表，完整演示 Skip-gram 模型中 softmax 概率的计算过程——从点积到指数化到归一化。## 和本讲哪个 waypoint 对应本 capsule 对应 **WP03: Word2Vec / Skip-gram / Softmax**。Slides p28-30 和 Notes §3.2 Eq.4-5 给出了 softmax 公式：$$P(o|c) = \frac{\exp(u_o^T v_c)}{\sum_{w \in V} \exp(u_w^T v_c)}$$但只看公式很难理解**分母的归一化到底在做什么**。这个 notebook 用一个 8 词的小词汇表，把每一步都算出来给你看。

## 设置：导入和词汇表我们只用 `numpy`（Colab 预装）。词汇表故意选了两个语义簇：- **金融**：banking, money, credit, loan- **河流**：river, stream, water- **歧义词**：bank（可以是银行也可以是河岸）

In [ ]:
import numpy as npnp.random.seed(42)  # 固定种子，每次运行结果一样# 8 个词的微型词汇表vocab = ["banking", "money", "river", "bank", "stream", "credit", "loan", "water"]vocab_size = len(vocab)word_to_idx = {w: i for i, w in enumerate(vocab)}d = 4  # 嵌入维度（toy example，4 维足够演示）print(f"词汇表 |V|={vocab_size}: {vocab}")print(f"嵌入维度 d={d}")

## Part A: 随机初始化（训练前）先看看**训练前**的 softmax 是什么样的。随机初始化 center 向量 V 和 context 向量 U，然后：1. 选 "banking" 作为 center word2. 算它和所有 context words 的点积3. 做 softmax 得到概率**运行后先看哪里**：看最后的概率列——是不是所有词的概率都差不多？

In [ ]:
# 随机初始化V = np.random.randn(vocab_size, d) * 0.1  # center vectorsU = np.random.randn(vocab_size, d) * 0.1  # context vectorscenter_word = "banking"c_idx = word_to_idx[center_word]v_c = V[c_idx]# 点积: score[w] = u_w^T v_cscores = U @ v_c# Softmax（数值稳定版：减去最大值）scores_shifted = scores - np.max(scores)exp_scores = np.exp(scores_shifted)probs = exp_scores / np.sum(exp_scores)# 打印表格print(f"{'Word':<12} {'Dot Score':>12} {'P(o|c)':>12}")print("-" * 38)for i, w in enumerate(vocab):    marker = " <- center" if w == center_word else ""    print(f"  {w:<10} {scores[i]:>12.6f} {probs[i]:>12.6f}{marker}")print(f"\n  sum P(o|c) = {np.sum(probs):.10f}")print(f"\n  最高/最低比值: {np.max(probs)/np.min(probs):.2f}x")

### 输出怎么解释**随机初始化下，所有词的概率几乎一样（约 1/8 = 0.125）。**这是因为：- 随机向量的点积都接近 0（高维空间中随机向量近似正交）- exp(≈0) ≈ 1，所以所有 exp 值差不多- 归一化后每个概率 ≈ 1/|V|**这说明训练前模型完全没有区分能力——它不知道哪些词更可能出现在 "banking" 附近。**

## Part B: 手工设计的"训练后"向量现在我们手工设计一组向量来**模拟训练后的效果**：- 金融词在第 0 维有较大的正值- 河流词在第 1 维有较大的正值- bank（歧义词）在两维都有中等值**运行后先看哪里**：对比 Part A 的概率——金融词的概率是不是明显升高了？

In [ ]:
# 手工设计的"训练后"向量V_train = np.array([    [1.0, 0.0, 0.2, 0.1],   # banking  - finance    [0.8, 0.1, 0.1, 0.0],   # money    - finance    [0.0, 1.0, 0.2, 0.1],   # river    - river    [0.5, 0.5, 0.1, 0.1],   # bank     - ambiguous    [0.1, 0.9, 0.1, 0.2],   # stream   - river    [0.9, 0.0, 0.1, 0.1],   # credit   - finance    [0.7, 0.1, 0.0, 0.2],   # loan     - finance    [0.0, 0.8, 0.1, 0.3],   # water    - river], dtype=np.float64)U_train = V_train.copy()v_c_train = V_train[c_idx]scores_train = U_train @ v_c_train# Softmaxscores_shifted = scores_train - np.max(scores_train)exp_scores = np.exp(scores_shifted)probs_train = exp_scores / np.sum(exp_scores)print(f"{'Word':<12} {'Dot Score':>12} {'P(o|c)':>12}")print("-" * 38)for i, w in enumerate(vocab):    marker = " <- center" if w == center_word else ""    print(f"  {w:<10} {scores_train[i]:>12.6f} {probs_train[i]:>12.6f}{marker}")print(f"\n  最高/最低比值: {np.max(probs_train)/np.min(probs_train):.2f}x")

### 输出怎么解释**训练后，概率分布不再均匀：**- 金融词（money, credit, loan）的概率明显升高- 河流词（river, stream, water）的概率明显降低- bank（歧义词）的概率居中**这就是 softmax 的核心作用**：把点积分数转化为概率分布。点积越大 → exp 越大 → 概率越高。**点积计算细节**（以 money 为例）：```u_money = [0.8, 0.1, 0.1, 0.0]v_banking = [1.0, 0.0, 0.2, 0.1]u_money · v_banking = 0.8×1.0 + 0.1×0.0 + 0.1×0.2 + 0.0×0.1 = 0.82```

## 可视化：概率分布下面的柱状图展示了训练后的 P(o|c)。**先看哪个轴**：x 轴是 context word，y 轴是概率。虚线是均匀分布 1/8 = 0.125。**哪个数值最关键**：金融词（蓝色）的柱子明显高于虚线，河流词（绿色）明显低于虚线。**它支持哪个课堂结论**：训练后的词向量能让 softmax 给语义相关的词分配更高的概率。

In [ ]:
import matplotlib.pyplot as pltfrom matplotlib.patches import Patch# 颜色方案colors = []for w in vocab:    if w == center_word:        colors.append("#e74c3c")  # red = center    elif w in ["banking", "money", "credit", "loan"]:        colors.append("#3498db")  # blue = finance    elif w in ["river", "stream", "water"]:        colors.append("#2ecc71")  # green = river    else:        colors.append("#f39c12")  # orange = ambiguousfig, ax = plt.subplots(figsize=(10, 5))bars = ax.bar(vocab, probs_train, color=colors, edgecolor="#2c3e50")for bar, prob in zip(bars, probs_train):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,            f"{prob:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")ax.set_xlabel("Context word (o)", fontsize=12)ax.set_ylabel("P(o | c)", fontsize=12)ax.set_title(f"Skip-gram Softmax: P(o | c='{center_word}') -- Trained", fontsize=12)ax.set_ylim(0, max(probs_train) * 1.3)ax.axhline(y=1.0/vocab_size, color="gray", linestyle="--", alpha=0.7)ax.text(7.5, 1.0/vocab_size + 0.003, f"uniform=1/8=0.125", fontsize=8, color="gray", ha="right")legend = [    Patch(facecolor="#e74c3c", label=f"center ({center_word})"),    Patch(facecolor="#3498db", label="finance"),    Patch(facecolor="#2ecc71", label="river"),    Patch(facecolor="#f39c12", label="ambiguous"),]ax.legend(handles=legend, loc="upper right")plt.tight_layout()plt.show()

## 对比：随机 vs 训练后**容易误解的地方**：不要以为 softmax 本身在"学习"。Softmax 只是一个数学函数——把分数变成概率。真正在学习的是向量 U 和 V（通过梯度下降调整，见 WP04）。这个对比图展示了同一组词在随机初始化和"训练后"的概率变化。

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))x = np.arange(vocab_size)width = 0.35ax.bar(x - width/2, probs, width, label="Random init", color="#95a5a6")ax.bar(x + width/2, probs_train, width, label="Trained", color="#3498db")ax.set_xlabel("Context word (o)", fontsize=12)ax.set_ylabel("P(o | c)", fontsize=12)ax.set_title(f"Random vs Trained: P(o | c='{center_word}')", fontsize=13)ax.set_xticks(x)ax.set_xticklabels(vocab)ax.legend(fontsize=10)ax.axhline(y=1.0/vocab_size, color="gray", linestyle="--", alpha=0.7)plt.tight_layout()plt.show()print("\n对比总结:")print(f"{'Word':<12} {'P_random':>10} {'P_trained':>10} {'Delta':>10}")print("-" * 44)for i, w in enumerate(vocab):    diff = probs_train[i] - probs[i]    arrow = "^" if diff > 0.005 else ("v" if diff < -0.005 else "~")    print(f"  {w:<10} {probs[i]:>10.4f} {probs_train[i]:>10.4f} {diff:>+8.4f} {arrow}")

## 总结：这个 capsule 学到了什么1. **Softmax 的计算步骤**：点积 → 减 max（数值稳定）→ exp → 归一化2. **分母的作用**：把所有 exp 值归一化成概率分布（和为 1）3. **训练前 vs 训练后**：随机初始化时概率均匀；训练后语义相关词的概率升高4. **计算量问题**：分母要算 |V| 个 exp——这就是为什么 WP04 要引入 Negative Sampling### 和 Slides/Notes 的对应| 本 capsule | 官方材料 ||---|---|| 点积 u_o^T v_c | Slides p28, Notes Eq.4 分子 || Softmax 归一化 | Slides p30, Notes Eq.4 分母 || 训练目标 | Notes Eq.5: min E[-log P(o|c)] || 计算量问题 | WP04 的 Negative Sampling |### 下一步- 如果理解了 softmax，继续 WP04：目标函数和梯度- 如果想问 Hermes，可以问："softmax 的分母为什么叫 partition function？"